In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())


/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [2]:
batch_cfcheck_data = CF_OUTPUTS / "xgboost_genetic_2026-04-17" / "genetic_annotated_hltprhc.csv"
batch_cfcheck_df = pd.read_csv(batch_cfcheck_data)

In [ ]:
pd.set_option("display.max_rows", None)

In [ ]:
batch_cfcheck_df = batch_cfcheck_df.drop(columns=["hltprhc", "target_risk"])

In [ ]:

batch_cfcheck_df["cf_id"] = batch_cfcheck_df["cf_id"].replace({"original": "xin"})

batch_cfcheck_df = batch_cfcheck_df.rename(columns={
    "original_risk": "risk_before",
    "predicted_risk" : "predicted_risk_after",
    "meets_target_risk" : "valid",
})

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.986591,0.0,0.025955,NaN,NaN,1.22
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,18.986590,NaN,0.025955,0.012272,True,NaN
2,0,cf_1,NaN,1.0,5.0,NaN,1.0,NaN,18.986590,NaN,0.025955,0.002570,True,NaN
3,0,cf_1,NaN,NaN,6.0,7.0,NaN,NaN,18.974530,NaN,0.025955,0.142825,False,NaN


In [ ]:
# correct_cf_idx
# Fix cf_id indexing - group by query_index and number counterfactuals sequentially
def fix_cf_id(df):
    # For each query_index group
    fixed_rows = []
    for query_idx, group in df.groupby('query_index', sort=False):
        group = group.copy()
        # First row is xin (original)
        xin_mask = group['cf_id'] == 'xin'
        cf_mask = ~xin_mask

        # Keep xin rows as is
        xin_rows = group[xin_mask].copy()

        # Fix cf rows
        cf_rows = group[cf_mask].copy()
        if len(cf_rows) > 0:
            # Number them cf_1, cf_2, cf_3, etc.
            cf_rows['cf_id'] = [f'cf_{i+1}' for i in range(len(cf_rows))]

        # Combine back
        fixed_rows.append(xin_rows)
        if len(cf_rows) > 0:
            fixed_rows.append(cf_rows)

    return pd.concat(fixed_rows, ignore_index=True)

batch_cfcheck_df = fix_cf_id(batch_cfcheck_df)
batch_cfcheck_df.head(30)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.986591,0.0,0.025955,NaN,NaN,1.22
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,18.986590,NaN,0.025955,0.012272,True,NaN
2,0,cf_2,NaN,1.0,5.0,NaN,1.0,NaN,18.986590,NaN,0.025955,0.002570,True,NaN
3,0,cf_3,NaN,NaN,6.0,7.0,NaN,NaN,18.974530,NaN,0.025955,0.142825,False,NaN
4,0,cf_4,NaN,NaN,5.0,NaN,NaN,NaN,19.031140,4.0,0.025955,0.008059,True,NaN
5,0,cf_5,2.0,4.0,6.0,1.0,NaN,NaN,18.986590,NaN,0.025955,0.048545,False,NaN
6,0,cf_6,2.0,1.0,5.0,NaN,1.0,NaN,18.986590,NaN,0.025955,0.003719,True,NaN
7,0,cf_7,6.0,1.0,5.0,7.0,NaN,NaN,18.986590,NaN,0.025955,0.001736,True,NaN
8,0,cf_8,2.0,NaN,6.0,5.0,NaN,NaN,18.986590,5.0,0.025955,0.079886,False,NaN
9,0,cf_9,1.0,1.0,4.0,NaN,1.0,NaN,18.986590,NaN,0.025955,0.017961,False,NaN


In [ ]:
# round BMI to 2 decimal places, and set to NaN if equal to xin after rounding
def round_and_clear_bmi(df):
    df = df.copy()

    # Round BMI to 2 decimals for all rows
    df['bmi'] = df['bmi'].round(2)

    # For each query_index, get the xin BMI and compare
    for query_idx in df['query_index'].unique():
        query_mask = df['query_index'] == query_idx
        xin_mask = query_mask & (df['cf_id'] == 'xin')
        cf_mask = query_mask & (df['cf_id'] != 'xin')

        # Get the xin BMI value for this query
        xin_bmi = df.loc[xin_mask, 'bmi'].values[0]

        # Set cf BMI to NaN where it equals xin BMI
        equal_mask = cf_mask & (df['bmi'] == xin_bmi)
        df.loc[equal_mask, 'bmi'] = np.nan

    return df

batch_cfcheck_df = round_and_clear_bmi(batch_cfcheck_df)
batch_cfcheck_df.head(30)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.99,0.0,0.025955,NaN,NaN,1.22
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.025955,0.012272,True,NaN
2,0,cf_2,NaN,1.0,5.0,NaN,1.0,NaN,NaN,NaN,0.025955,0.002570,True,NaN
3,0,cf_3,NaN,NaN,6.0,7.0,NaN,NaN,18.97,NaN,0.025955,0.142825,False,NaN
4,0,cf_4,NaN,NaN,5.0,NaN,NaN,NaN,19.03,4.0,0.025955,0.008059,True,NaN
5,0,cf_5,2.0,4.0,6.0,1.0,NaN,NaN,NaN,NaN,0.025955,0.048545,False,NaN
6,0,cf_6,2.0,1.0,5.0,NaN,1.0,NaN,NaN,NaN,0.025955,0.003719,True,NaN
7,0,cf_7,6.0,1.0,5.0,7.0,NaN,NaN,NaN,NaN,0.025955,0.001736,True,NaN
8,0,cf_8,2.0,NaN,6.0,5.0,NaN,NaN,NaN,5.0,0.025955,0.079886,False,NaN
9,0,cf_9,1.0,1.0,4.0,NaN,1.0,NaN,NaN,NaN,0.025955,0.017961,False,NaN


In [ ]:
int_columns = [
    "etfruit",
    "eatveg",
    "cgtsmok",
    "alcfreq",
    "slprl",
    "paccnois",
    "dosprt",
]

float_columns = [
    "bmi",
    "risk_before",
    "predicted_risk_after",
]

In [ ]:
batch_cfcheck_df[int_columns] = batch_cfcheck_df[int_columns].astype("Int64")

In [ ]:
batch_cfcheck_df[float_columns] = batch_cfcheck_df[float_columns].round(2)

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,0.03,NaN,NaN,1.22
1,0,cf_1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,0.03,0.01,True,NaN
2,0,cf_2,<NA>,1,5,<NA>,1,<NA>,NaN,<NA>,0.03,0.00,True,NaN
3,0,cf_3,<NA>,<NA>,6,7,<NA>,<NA>,18.97,<NA>,0.03,0.14,False,NaN


In [ ]:
batch_cfcheck_df[int_columns] = (
    batch_cfcheck_df[int_columns]
    .astype("string")
    .fillna("")
)

In [ ]:
batch_cfcheck_df = batch_cfcheck_df.replace({np.nan : ""})
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,0.03,,,1.22
1,0,cf_1,,,,,,,,,0.03,0.01,True,
2,0,cf_2,,1,5,,1,,,,0.03,0.0,True,
3,0,cf_3,,,6,7,,,18.97,,0.03,0.14,False,


In [ ]:
mask = batch_cfcheck_df["cf_id"] == "xin"
batch_cfcheck_df.loc[mask, "risk_before"] = ""
batch_cfcheck_df.head(4)

/tmp/ipykernel_39625/1160248817.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  batch_cfcheck_df.loc[mask, "risk_before"] = ""


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,,,,1.22
1,0,cf_1,,,,,,,,,0.03,0.01,True,
2,0,cf_2,,1,5,,1,,,,0.03,0.0,True,
3,0,cf_3,,,6,7,,,18.97,,0.03,0.14,False,


In [ ]:
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# 1. Beräkna Nchanged endast för CF-rader
mask_cf = batch_cfcheck_df["cf_id"] != "xin"

batch_cfcheck_df.loc[mask_cf, "Nchanged"] = (
    batch_cfcheck_df.loc[mask_cf, feature_cols] != ""
).sum(axis=1)

# 2. Konvertera kolumnen till ren string
batch_cfcheck_df["Nchanged"] = batch_cfcheck_df["Nchanged"].astype("string")

# 3. Baseline ska vara tom
batch_cfcheck_df.loc[batch_cfcheck_df["cf_id"] == "xin", "Nchanged"] = ""

In [ ]:
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec,Nchanged
0,0,xin,4,3,3,4,2,0,18.99,0,,,,1.22,
1,0,cf_1,,,,,,,,,0.03,0.01,True,,0
2,0,cf_2,,1,5,,1,,,,0.03,0.0,True,,3
3,0,cf_3,,,6,7,,,18.97,,0.03,0.14,False,,3


In [ ]:
batch_cfcheck_df["Gower"] = ""
batch_cfcheck_df["Expected"] = ""


In [ ]:
order = ["query_index", "cf_id"] + feature_cols + ["cf_gen_time_sec", "Gower", "Nchanged", "valid", "risk_before", "predicted_risk_after", "Expected"]

batch_cfcheck_df = batch_cfcheck_df[order]
batch_cfcheck_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,1.22,,,,,,
1,0,cf_1,,,,,,,,,,,0,True,0.03,0.01,
2,0,cf_2,,1,5,,1,,,,,,3,True,0.03,0.0,
3,0,cf_3,,,6,7,,,18.97,,,,3,False,0.03,0.14,
4,0,cf_4,,,5,,,,19.03,4,,,3,True,0.03,0.01,
5,0,cf_5,2,4,6,1,,,,,,,4,False,0.03,0.05,
6,0,cf_6,2,1,5,,1,,,,,,4,True,0.03,0.0,
7,0,cf_7,6,1,5,7,,,,,,,4,True,0.03,0.0,
8,0,cf_8,2,,6,5,,,,5,,,4,False,0.03,0.08,
9,0,cf_9,1,1,4,,1,,,,,,4,False,0.03,0.02,


# picking correct CF

### expected


In [ ]:
# Check directional constraints directly
# These features should only improve (increase) or stay the same
SHOULD_INCREASE = ["cgtsmok", "alcfreq", "dosprt"]

# These features should only improve (decrease) or stay the same
SHOULD_DECREASE = ["bmi", "etfruit", "eatveg", "slprl", "paccnois"]


print("Directional constraints:")
print(f"  Should increase (↑): {SHOULD_INCREASE}")
print(f"  Should decrease (↓): {SHOULD_DECREASE}")


Directional constraints:
  Should increase (↑): ['cgtsmok', 'alcfreq', 'dosprt']
  Should decrease (↓): ['bmi', 'etfruit', 'eatveg', 'slprl', 'paccnois']


In [ ]:
# Check if CFs violate directional constraints
# First, ensure numeric columns are actually numeric for comparison
numeric_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# Create a working copy with numeric values
df_check = batch_cfcheck_df.copy()
for col in numeric_cols:
    df_check[col] = pd.to_numeric(df_check[col], errors="coerce")

# Extract baseline values per query_index
baseline_values = df_check[df_check["cf_id"] == "xin"].set_index("query_index")[numeric_cols]

# Check violations for each CF
def check_violations(row):
    if row["cf_id"] == "xin":
        return ""

    baseline = baseline_values.loc[row["query_index"]]
    violations = []

    # Check features that should increase
    for feat in SHOULD_INCREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] < baseline[feat]:
                violations.append(f"{feat}↓")

    # Check features that should decrease
    for feat in SHOULD_DECREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] > baseline[feat]:
                violations.append(f"{feat}↑")

    return "NO: " + ", ".join(violations) if violations else ""

batch_cfcheck_df["Expected"] = df_check.apply(check_violations, axis=1)
batch_cfcheck_df


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,1.22,,,,,,
1,0,cf_1,,,,,,,,,,,0,True,0.03,0.01,
2,0,cf_2,,1,5,,1,,,,,,3,True,0.03,0.0,
3,0,cf_3,,,6,7,,,18.97,,,,3,False,0.03,0.14,
4,0,cf_4,,,5,,,,19.03,4,,,3,True,0.03,0.01,NO: bmi↑
5,0,cf_5,2,4,6,1,,,,,,,4,False,0.03,0.05,"NO: alcfreq↓, eatveg↑"
6,0,cf_6,2,1,5,,1,,,,,,4,True,0.03,0.0,
7,0,cf_7,6,1,5,7,,,,,,,4,True,0.03,0.0,NO: etfruit↑
8,0,cf_8,2,,6,5,,,,5,,,4,False,0.03,0.08,
9,0,cf_9,1,1,4,,1,,,,,,4,False,0.03,0.02,


### is valid

In [ ]:
# --- Select exactly one CF per query_index (random valid without violations) ---

# Split baseline and CF rows
df_xin = batch_cfcheck_df[batch_cfcheck_df["cf_id"] == "xin"]
df_cf  = batch_cfcheck_df[batch_cfcheck_df["cf_id"] != "xin"]

def select_random_cf(group):
    """Select one random CF that is valid and has no violations (Expected is empty)."""
    # First try: valid AND no violations
    good_cfs = group[(group["valid"] == True) & (group["Expected"] == "")]
    if len(good_cfs) > 0:
        return good_cfs.sample(n=1, random_state=42)

    # Fallback: any valid CF
    valid_cfs = group[group["valid"] == True]
    if len(valid_cfs) > 0:
        return valid_cfs.sample(n=1, random_state=42)

    # Last resort: first CF
    return group.iloc[[0]]

# Select one random CF per query_index
df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)

# Combine baseline + selected CF
batch_cfcheck_df = pd.concat([df_xin, df_cf_selected], ignore_index=True)

# Ensure xin appears before CF for each query_index
batch_cfcheck_df["is_xin"] = (batch_cfcheck_df["cf_id"] == "xin").astype(int)
batch_cfcheck_df = (
    batch_cfcheck_df
    .sort_values(["query_index", "is_xin"], ascending=[True, False])
    .drop(columns="is_xin")
)
batch_cfcheck_df

/tmp/ipykernel_39625/2325069138.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,1.22,,,,,,
9,0,cf_1,,,,,,,,,,,0,True,0.03,0.01,
1,1,xin,3,4,1,2,3,0,22.38,0,1.3,,,,,,
10,1,cf_3,2,,,,1,,,,,,2,True,0.25,0.04,
2,2,xin,5,3,1,1,4,0,22.69,7,1.08,,,,,,
11,2,cf_3,1,,,,2,,22.68,,,,3,True,0.22,0.07,
3,3,xin,3,3,6,6,2,0,24.34,1,1.18,,,,,,
12,3,cf_1,,,,,,,,,,,0,False,0.07,0.07,
4,4,xin,5,4,2,7,1,0,21.26,3,1.31,,,,,,
13,4,cf_2,,2,5,,,,,,,,2,True,0.08,0.01,


In [ ]:
batch_cfcheck_df["valid"] = batch_cfcheck_df["valid"].replace(
    {
        False : "No",
        True : "Yes",
    }
)